In [ ]:
# USED TO SETUP SEGMENTATION INSIDE THE PROJECT
%load_ext autoreload
%autoreload 2

import random
import sys
from pathlib import Path

import cv2
import improutils as iu


def find_repo_root(start):
    """Locate repository root by expected project markers."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml/src)")

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Notebook execution has no package parent, so use absolute import from src root.
from preprocessing.segmentation.segmentation import build_masks

# --- Configuration ---
TREE_NUMBER = 11
TREE_NAME = f"dub{TREE_NUMBER}"
PNG_DIR = REPO_ROOT / "src" / "png" / TREE_NAME

FROM_SLICE = 5  # Lower bound for random selection
FORCE_SLICE = 200  # Set to int for fixed slice, or None for random


def pick_slice_file(png_dir, from_slice, force_slice):
    """Select one input slice either by fixed id or randomly from valid range."""
    if force_slice is not None:
        forced = png_dir / f"slice_{force_slice:04d}.png"
        if not forced.exists():
            raise FileNotFoundError(f"Forced slice not found: {forced}")
        print(f"Forced selection {TREE_NAME}: {forced.name}")
        return forced

    valid_files = []
    for path in sorted(png_dir.glob("*.png")):
        try:
            slice_num = int(path.stem.split("_")[-1])
        except ValueError:
            continue
        if slice_num >= from_slice:
            valid_files.append(path)

    if not valid_files:
        raise ValueError(f"No valid files found starting from slice {from_slice}")

    selected = random.choice(valid_files)
    print(f"Random selection: {selected.name} {TREE_NAME}")
    return selected


target_file = pick_slice_file(PNG_DIR, FROM_SLICE, FORCE_SLICE)

# --- Processing ---
img_bgr = cv2.imread(str(target_file))
if img_bgr is None:
    raise FileNotFoundError(f"Image could not be read: {target_file}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Valid mask names: pozadi, kura, suk, hniloba, trhlina
requested_masks = {"kura", "pozadi", "trhlina", "hniloba", "suk"}
results = build_masks(img, requested_masks=requested_masks)

kura = results["kura"]
pozadi = results["pozadi"]
trhlina = results["trhlina"]
hniloba = results["hniloba"]
suk = results["suk"]

iu.plot_images(
    img,
    kura,
    pozadi,
    trhlina,
    hniloba,
    suk,
    titles=[ "Original", "Kura", "Pozadi", "Trhlina", "Hniloba", "Suk"],
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


FileNotFoundError: Forced slice not found: c:\Users\olive\vscode\bp-main\src\png\dub12\slice_0200.png